In [ ]:
from transformers import BlipProcessor, BlipForConditionalGeneration, BlipForQuestionAnswering
from PIL import Image
import requests
image_url = "https://images.unsplash.com/photo-1519125323398-675f0ddb6308"
raw_image = Image.open(requests.get(image_url, stream=True).raw).convert("RGB")
# ---------- Image Captioning ----------
cap_processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
cap_model = BlipForConditionalGeneration.from_pretrained("Salesforce/blip-image-captioning-base")
inputs = cap_processor(raw_image, return_tensors="pt")
caption_ids = cap_model.generate(**inputs, max_new_tokens=30)
caption = cap_processor.decode(caption_ids[0], skip_special_tokens=True)
print("Generated Caption:", caption)
# ---------- Visual Question Answering ----------
vqa_processor = BlipProcessor.from_pretrained("Salesforce/blip-vqa-base")
vqa_model = BlipForQuestionAnswering.from_pretrained("Salesforce/blip-vqa-base")
question = "What animal is in the picture?"
vqa_inputs = vqa_processor(raw_image, question, return_tensors="pt")
answer_ids = vqa_model.generate(**vqa_inputs)
answer = vqa_processor.decode(answer_ids[0], skip_special_tokens=True)
print("Question:", question)
print("Answer:", answer)

preprocessor_config.json:   0%|          | 0.00/287 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/4.56k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/506 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.54GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/788 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/transformers/generation/utils.py:1625: UserWarning: Using the model-agnostic default `max_length` (=21) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(


Question: What animal is in the picture?
Answer: bird


In [ ]:
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)
import numpy as np
from sklearn.metrics import accuracy_score

# 1. Install/upgrade required libraries
%pip install -q --upgrade transformers datasets huggingface_hub

# IMPORTANT:
# If the notebook asks you to restart the runtime,
# restart it and then run the code again from the imports.

# 2. Load IMDB dataset
dataset = load_dataset("stanfordnlp/imdb")

# Use smaller datasets for faster training
small_train = dataset["train"].shuffle(seed=42).select(range(2000))
small_test = dataset["test"].shuffle(seed=42).select(range(500))

print(dataset)
print("Training samples:", len(small_train))
print("Testing samples:", len(small_test))

# 3. Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(
    "distilbert-base-uncased"
)

# Tokenization function
def tokenize(batch):
    return tokenizer(
        batch["text"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

# Tokenize datasets
train_ds = small_train.map(tokenize, batched=True)
test_ds = small_test.map(tokenize, batched=True)

# 4. Load pre-trained model
model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=2
)

# 5. Training arguments
args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=2,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    eval_strategy="epoch",
    logging_steps=50
)

# 6. Calculate accuracy
def compute_metrics(pred):
    preds = np.argmax(pred.predictions, axis=1)

    return {
        "accuracy": accuracy_score(
            pred.label_ids,
            preds
        )
    }

# 7. Create Trainer
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=test_ds,
    compute_metrics=compute_metrics
)

# 8. Train
trainer.train()

# 9. Evaluate
metrics = trainer.evaluate()

print("Evaluation metrics:", metrics)

# 10. Save fine-tuned model
model.save_pretrained("./fine_tuned_distilbert_imdb")
tokenizer.save_pretrained("./fine_tuned_distilbert_imdb")

print("Model saved successfully!")

In [ ]:
from transformers import pipeline
from diffusers import StableDiffusionPipeline
from gtts import gTTS
import torch
topic = "The benefits of renewable energy"
# 1. Text generation
text_generator = pipeline("text2text-generation", model="google/flan-t5-base")
text_prompt = f"Write a short, engaging paragraph about: {topic}"
generated_text = text_generator(text_prompt, max_length=80)[0]["generated_text"]
print("Generated Text:\n", generated_text)
# 2. Image generation (derived prompt)
image_prompt = f"An illustration representing {topic}, digital art"
sd_pipe = StableDiffusionPipeline.from_pretrained(
"runwayml/stable-diffusion-v1-5", torch_dtype=torch.float16).to("cuda")
image = sd_pipe(image_prompt, num_inference_steps=25).images[0]
image.save("content_image.png")
print("Image saved as content_image.png")
# 3. Audio generation (text-to-speech)
tts = gTTS(text=generated_text, lang="en")
tts.save("content_audio.mp3")
print("Audio saved as content_audio.mp3")

In [ ]:
import gradio as gr
from transformers import pipeline
import evaluate
# ---------- 1. Build and Deploy the App ----------
summarizer = pipeline("summarization", model="facebook/bart-large-cnn")
def summarize_text(input_text):
result = summarizer(input_text, max_length=45, min_length=15, do_sample=False)
return result[0]["summary_text"]
demo = gr.Interface(
fn=summarize_text,
inputs=gr.Textbox(lines=8, label="Enter text to summarize"),
outputs=gr.Textbox(label="Generated Summary"),
title="GenAI Text Summarizer",
description="A cloud-deployable Generative AI summarization app built with Gradio."
)
demo.launch(share=True) # share=True generates a public cloud URL
# ---------- 2. Evaluate Generated Output ----------
rouge = evaluate.load("rouge")
generated_summaries = [
"AI models generate new content such as text and images.",
]
reference_summaries = [
"Generative AI models are capable of producing new content including text and images.",
]
scores = rouge.compute(predictions=generated_summaries, references=reference_summaries)
print("ROUGE Evaluation Scores:", scores)